# QIP Material Intelligence — Google Colab

Run construction-material screening **as code** (no web frontend).

**Flow**
1. Upload image (Brick / Steel / Cement / Wood)
2. Preprocessing — CLAHE · Resize · Normalize
3. AI analysis — visual + condition heuristics
4. Condition — Good / Damaged / Inspection Required
5. Recommendation — Routine checks / Further inspection / Hold for review
6. Results table (batch) you can download as CSV

**Runtime:** Runtime → Change runtime type → CPU is enough.

## 0) Install & clone project

In [ ]:
# Install light deps (no full Torch/YOLO needed for Colab heuristic scan)
%pip install -q opencv-python-headless pillow numpy pandas matplotlib python-dotenv pydantic pydantic-settings

import os, sys
from pathlib import Path

REPO = "https://github.com/dleelaprasad-2665/QIP-PROJECT.git"
ROOT = Path("/content/QIP-PROJECT")

if not ROOT.exists():
    !git clone --depth 1 {REPO} {ROOT}
else:
    print("Repo already present:", ROOT)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working dir:", Path.cwd())

## 1) Load scanner

In [ ]:
from dataclasses import asdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
from IPython.display import display, Markdown

from material_intelligence.app.ml.colab_scan import scan_image, scan_path, scan_folder, load_bgr

def show_bgr(image_bgr, title=""):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(7, 5))
    plt.imshow(rgb)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

def print_result(result):
    display(Markdown(f"""
### Result
| Step | Value |
|---|---|
| **1. Material** | **{result.material}** |
| **2. Preprocessing** | {result.preprocess} |
| **3. AI notes** | {result.ai_notes} |
| **4. Condition** | **{result.condition}** ({result.confidence_pct}%) |
| **5. Recommendation** | **{result.recommendation}** |
| **Details** | {result.detail} |
"""))

print("Scanner ready.")

## 2) Upload one photo and scan

Set `MATERIAL` to `auto`, `brick`, `steel`, `cement`, or `wood`.

In [ ]:
MATERIAL = "auto"  # auto | brick | steel | cement | wood

uploaded = files.upload()
assert uploaded, "Upload at least one image."

name = next(iter(uploaded))
path = Path("/content") / name
path.write_bytes(uploaded[name])

image = load_bgr(path)
show_bgr(image, title=f"1. Upload — {name}")

result = scan_path(path, material=MATERIAL)
print_result(result)

## 3) Batch scan many photos

Upload several images (or a zip). Results are shown in a table and saved to CSV.

In [ ]:
import io, zipfile

MATERIAL = "auto"
batch_dir = Path("/content/batch_uploads")
batch_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for fname, raw in uploaded.items():
    p = Path(fname)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            zf.extractall(batch_dir)
    elif p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        (batch_dir / p.name).write_bytes(raw)

rows = scan_folder(batch_dir, material=MATERIAL)
df = pd.DataFrame(rows)
if df.empty:
    print("No images found.")
else:
    cols = ["file", "material", "condition", "recommendation", "confidence_pct", "detail"]
    display(df[cols])
    out_csv = Path("/content/batch_results.csv")
    df[cols].to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print(
        f"Summary — Good: {(df.condition=='Good').sum()} · "
        f"Damaged: {(df.condition=='Damaged').sum()} · "
        f"Inspection Required: {(df.condition=='Inspection Required').sum()}"
    )
    files.download(str(out_csv))

## 4) Optional — scan a Drive folder

Mount Google Drive, then set `DRIVE_FOLDER` to your photos folder.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# DRIVE_FOLDER = "/content/drive/MyDrive/your_material_photos"
# MATERIAL = "auto"
#
# rows = scan_folder(DRIVE_FOLDER, material=MATERIAL)
# df = pd.DataFrame(rows)
# display(df[["file", "material", "condition", "recommendation", "confidence_pct"]])
# df.to_csv("/content/drive_batch_results.csv", index=False)
print("Uncomment the lines above to scan a Drive folder.")

## Notes

- This Colab path uses the same **material + condition heuristics** as the app field checks (rust, brick edges, bag packaging).
- The NiceGUI website is separate (`python -m material_intelligence.app.main`) — not required here.
- For YOLO/EfficientNet weights on Colab, upload `data/models/...` and we can extend this notebook to load them.